In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

folder = "/content/drive/MyDrive/Fraud_Project"

print(os.listdir(folder))

['firm_years.json', 'financial_with_aaer_labels.csv', 'aaer_label_quality_check.csv', 'clean_mda.csv', 'matched_mda.csv']


In [ ]:
import pandas as pd
import os

folder = "/content/drive/MyDrive/Fraud_Project"

financial = pd.read_csv(
    os.path.join(folder, "financial_with_aaer_labels.csv")
)

mda = pd.read_csv(
    os.path.join(folder, "clean_mda.csv")
)

print("Financial rows:", len(financial))
print("Clean MD&A rows:", len(mda))

Financial rows: 40225
Clean MD&A rows: 31241


In [ ]:
import pandas as pd
import os

folder = "/content/drive/MyDrive/Fraud_Project"

financial_file = os.path.join(
    folder, "financial_with_aaer_labels.csv"
)

mda_file = os.path.join(
    folder, "clean_mda.csv"
)

# Load datasets
financial = pd.read_csv(financial_file)
mda = pd.read_csv(mda_file)

# ---------------------------------------------------------
# STANDARDIZE CIK
# ---------------------------------------------------------

financial["cik"] = pd.to_numeric(
    financial["cik"], errors="coerce"
).astype("Int64")

mda["cik"] = pd.to_numeric(
    mda["cik"], errors="coerce"
).astype("Int64")

# ---------------------------------------------------------
# STANDARDIZE YEAR
# ---------------------------------------------------------

financial["fiscal_year"] = pd.to_numeric(
    financial["fiscal_year"], errors="coerce"
).astype("Int64")

mda["fiscal_year"] = pd.to_numeric(
    mda["fiscal_year"], errors="coerce"
).astype("Int64")

# ---------------------------------------------------------
# CREATE MATCHING KEY
# ---------------------------------------------------------

financial["cik_year"] = (
    financial["cik"].astype(str)
    + "_"
    + financial["fiscal_year"].astype(str)
)

mda["cik_year"] = (
    mda["cik"].astype(str)
    + "_"
    + mda["fiscal_year"].astype(str)
)

# ---------------------------------------------------------
# FIND EXACT OVERLAP
# ---------------------------------------------------------

financial_keys = set(financial["cik_year"])
mda_keys = set(mda["cik_year"])

matched_keys = financial_keys.intersection(mda_keys)

print("======================================")
print("DATASET MATCHING RESULTS")
print("======================================")

print("Financial company-years :", len(financial_keys))
print("Clean MD&A company-years:", len(mda_keys))
print("Matched company-years   :", len(matched_keys))

coverage = len(matched_keys) / len(financial_keys) * 100

print(
    f"Financial records with MD&A: {coverage:.2f}%"
)

# ---------------------------------------------------------
# CREATE FINAL MERGED DATASET
# ---------------------------------------------------------

final_data = financial[
    financial["cik_year"].isin(matched_keys)
].copy()

# Keep only necessary MD&A columns
mda_small = mda[
    ["cik_year", "reporting_date", "mda", "url", "mda_length"]
].copy()

# Merge
final_data = final_data.merge(
    mda_small,
    on="cik_year",
    how="inner",
    validate="one_to_one"
)

# ---------------------------------------------------------
# REMOVE HELPER COLUMN
# ---------------------------------------------------------

final_data = final_data.drop(
    columns=["cik_year"]
)

# ---------------------------------------------------------
# SAVE
# ---------------------------------------------------------

output_file = os.path.join(
    folder,
    "final_financial_mda_dataset.csv"
)

final_data.to_csv(
    output_file,
    index=False
)

print("\n======================================")
print("FINAL DATASET")
print("======================================")

print("Rows:", len(final_data))
print("Columns:", len(final_data.columns))

print("\nSaved to:")
print(output_file)

# ---------------------------------------------------------
# FRAUD LABEL DISTRIBUTION
# ---------------------------------------------------------

print("\n======================================")
print("AAER LABEL DISTRIBUTION")
print("======================================")

print(
    final_data["is_fraud"]
    .value_counts(dropna=False)
)

print("\nFraud percentage:")

print(
    final_data["is_fraud"].mean() * 100
)

# ---------------------------------------------------------
# CHECK
# ---------------------------------------------------------

print("\nDuplicate CIK-year pairs:")

print(
    final_data.duplicated(
        subset=["cik", "fiscal_year"]
    ).sum()
)

display(final_data.head(3))

DATASET MATCHING RESULTS
Financial company-years : 40225
Clean MD&A company-years: 31241
Matched company-years   : 31241
Financial records with MD&A: 77.67%

FINAL DATASET
Rows: 31241
Columns: 23

Saved to:
/content/drive/MyDrive/Fraud_Project/final_financial_mda_dataset.csv

AAER LABEL DISTRIBUTION
is_fraud
0    30965
1      276
Name: count, dtype: int64

Fraud percentage:
0.8834544348772446

Duplicate CIK-year pairs:
0


,ticker,cik,fiscal_year,price,price_date,ddate,Assets,AssetsCurrent,CashAndCashEquivalentsAtCarryingValue,CommonStockSharesIssued,...,OperatingIncomeLoss,StockholdersEquity,NetIncomeLoss,Liabilities,is_fraud,fraud_label_source,reporting_date,mda,url,mda_length
0,A,1090872,2010,24.856937,2010-11-01,2010-10-31,9.696000e+09,6.169000e+09,2.649000e+09,579000000.0,...,5.660000e+08,3.228000e+09,NaN,6.460000e+09,0,No_AAER_fraud_period_overlap,31-10-2010,Item 7. Management's Discussion and Analysis ...,https://www.sec.gov/Archives/edgar/data/109087...,90792
1,A,1090872,2011,26.516453,2011-10-31,2011-10-31,9.057000e+09,5.569000e+09,3.527000e+09,591000000.0,...,1.071000e+09,4.308000e+09,NaN,4.741000e+09,0,No_AAER_fraud_period_overlap,31-10-2011,Item 7. Management's Discussion and Analysis ...,https://www.sec.gov/Archives/edgar/data/109087...,94076
2,A,1090872,2012,25.743919,2012-10-31,2012-10-31,1.053600e+10,4.629000e+09,2.351000e+09,595000000.0,...,1.119000e+09,5.182000e+09,NaN,5.351000e+09,0,No_AAER_fraud_period_overlap,31-10-2012,Item 7. Management's Discussion and Analysis ...,https://www.sec.gov/Archives/edgar/data/109087...,106109


In [ ]:
import pandas as pd
import os

from sklearn.model_selection import StratifiedGroupKFold

folder = "/content/drive/MyDrive/Fraud_Project"

file = os.path.join(
    folder,
    "final_financial_mda_dataset.csv"
)

df = pd.read_csv(file)

print("Total rows:", len(df))
print("Unique companies:", df["cik"].nunique())

print("\nClass distribution:")
print(df["is_fraud"].value_counts())

print("\nFraud percentage:")
print(df["is_fraud"].mean() * 100)

print("\nPositive companies:")
print(
    df.loc[df["is_fraud"] == 1, "cik"].nunique()
)

print("\nCompanies with both fraud and non-fraud years:")
company_labels = df.groupby("cik")["is_fraud"].nunique()

print(
    (company_labels > 1).sum()
)

Total rows: 31241
Unique companies: 2931

Class distribution:
is_fraud
0    30965
1      276
Name: count, dtype: int64

Fraud percentage:
0.8834544348772446

Positive companies:
80

Companies with both fraud and non-fraud years:
80


In [ ]:
# ---------------------------------------------------------
# COMPANY-LEVEL STRATIFIED SPLIT
# ---------------------------------------------------------

X = df.drop(columns=["is_fraud"])
y = df["is_fraud"]
groups = df["cik"]

sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

splits = list(
    sgkf.split(X, y, groups)
)

# First fold = test
train_val_idx, test_idx = splits[0]

train_val = df.iloc[train_val_idx].copy()
test = df.iloc[test_idx].copy()

# Split remaining data into train and validation
X_tv = train_val.drop(columns=["is_fraud"])
y_tv = train_val["is_fraud"]
groups_tv = train_val["cik"]

sgkf2 = StratifiedGroupKFold(
    n_splits=4,
    shuffle=True,
    random_state=42
)

train_idx, val_idx = next(
    sgkf2.split(
        X_tv,
        y_tv,
        groups_tv
    )
)

train = train_val.iloc[train_idx].copy()
validation = train_val.iloc[val_idx].copy()

print("TRAIN:", len(train))
print("VALIDATION:", len(validation))
print("TEST:", len(test))

print("\nTRAIN fraud distribution:")
print(train["is_fraud"].value_counts())

print("\nVALIDATION fraud distribution:")
print(validation["is_fraud"].value_counts())

print("\nTEST fraud distribution:")
print(test["is_fraud"].value_counts())

print("\nUnique companies:")
print("Train:", train["cik"].nunique())
print("Validation:", validation["cik"].nunique())
print("Test:", test["cik"].nunique())

print("\nCompany overlap checks:")

print(
    "Train ∩ Validation:",
    len(
        set(train["cik"]) &
        set(validation["cik"])
    )
)

print(
    "Train ∩ Test:",
    len(
        set(train["cik"]) &
        set(test["cik"])
    )
)

print(
    "Validation ∩ Test:",
    len(
        set(validation["cik"]) &
        set(test["cik"])
    )
)

TRAIN: 18734
VALIDATION: 6322
TEST: 6185

TRAIN fraud distribution:
is_fraud
0    18579
1      155
Name: count, dtype: int64

VALIDATION fraud distribution:
is_fraud
0    6265
1      57
Name: count, dtype: int64

TEST fraud distribution:
is_fraud
0    6121
1      64
Name: count, dtype: int64

Unique companies:
Train: 1761
Validation: 586
Test: 584

Company overlap checks:
Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0


In [ ]:
# ============================================================
# SAVE THE FINAL COMPANY-LEVEL SPLITS
# ============================================================

import os

base_path = "/content/drive/MyDrive/Fraud_Project"

train_path = os.path.join(base_path, "E1_train.csv")
val_path   = os.path.join(base_path, "E1_validation.csv")
test_path  = os.path.join(base_path, "E1_test.csv")

train.to_csv(train_path, index=False)
validation.to_csv(val_path, index=False)
test.to_csv(test_path, index=False)

print("Files saved:")
print(train_path)
print(val_path)
print(test_path)

# Verify
print("\nVerification:")
print("E1_train.csv:", os.path.exists(train_path))
print("E1_validation.csv:", os.path.exists(val_path))
print("E1_test.csv:", os.path.exists(test_path))


Files saved:
/content/drive/MyDrive/Fraud_Project/E1_train.csv
/content/drive/MyDrive/Fraud_Project/E1_validation.csv
/content/drive/MyDrive/Fraud_Project/E1_test.csv

Verification:
E1_train.csv: True
E1_validation.csv: True
E1_test.csv: True
